# 🏦 TA-03: Prediksi Risiko Gagal Bayar Kartu Kredit
## Regresi Logistik + SMOTE — Dataset UCI Credit Card

---

**Tujuan:** Membangun sistem prediksi kemungkinan nasabah gagal bayar kartu kredit bulan berikutnya.  
**Algoritma:** Logistic Regression  
**Dataset:** UCI Credit Card Default (30.000 nasabah, 25 variabel)  

### Alur Notebook
| # | Tahap |
|---|-------|
| 1 | Import Library |
| 2 | Muat Dataset |
| 3 | Eksplorasi Data (EDA) |
| 4 | Pra-pemrosesan Data |
| 5 | Penanganan Class Imbalance (SMOTE) |
| 6 | Training Model Regresi Logistik |
| 7 | Evaluasi Model |
| 8 | Simpan Model (Joblib) |
| 9 | Uji Prediksi Manual |


---
## ⚙️ Sel 1 — Import Library

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import json
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisasi ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Style visualisasi
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold'})

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve,
    accuracy_score,
)

# ── Imbalanced data ───────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── Simpan model ──────────────────────────────────────────────────────────────
import joblib

print('✅ Semua library berhasil diimpor!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   sklearn : ', end='')
import sklearn; print(sklearn.__version__)

---
## 📂 Sel 2 — Muat Dataset

In [ ]:
# ── Path dataset ──────────────────────────────────────────────────────────────
CSV_PATH   = 'dataset/UCI_Credit_Card.csv'   # sesuaikan jika perlu
MODEL_DIR  = 'model'                          # folder output model
EVAL_DIR   = os.path.join(MODEL_DIR, 'eval') # folder gambar evaluasi

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,  exist_ok=True)

# ── Muat CSV ──────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)

# UCI asli kadang punya baris header tambahan
if 'LIMIT_BAL' not in df_raw.columns:
    df_raw = pd.read_csv(CSV_PATH, header=1)

# Rename kolom agar konsisten
if 'PAY_1' in df_raw.columns:
    df_raw = df_raw.rename(columns={'PAY_1': 'PAY_0'})

target_col = next(c for c in df_raw.columns if 'default' in c.lower())
df_raw = df_raw.rename(columns={target_col: 'TARGET'})

# Drop kolom ID jika ada (tidak berguna untuk model)
if 'ID' in df_raw.columns:
    df_raw = df_raw.drop(columns=['ID'])

print(f'📊 Dataset dimuat: {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom')
df_raw.head()

---
## 🔍 Sel 3A — EDA: Informasi Dasar Dataset

In [ ]:
print('=' * 55)
print('INFORMASI UMUM DATASET')
print('=' * 55)
print(f'Jumlah baris    : {df_raw.shape[0]:,}')
print(f'Jumlah kolom    : {df_raw.shape[1]}')
print(f'Missing values  : {df_raw.isnull().sum().sum()}')
print()
df_raw.info()

In [ ]:
# Statistik deskriptif kolom numerik utama
df_raw[['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1', 'TARGET']].describe().round(2)

---
## 📊 Sel 3B — EDA: Distribusi Kelas Target (Class Imbalance)

In [ ]:
counts = df_raw['TARGET'].value_counts().sort_index()
labels = ['Tidak Gagal Bayar (0)', 'Gagal Bayar (1)']
colors = ['#1d7a4a', '#b3261e']

print('Distribusi Target:')
for i, (label, count) in enumerate(zip(labels, counts)):
    pct = count / len(df_raw) * 100
    print(f'  {label}: {count:,} ({pct:.1f}%)')
print(f'\nRasio ketidakseimbangan: 1 : {counts[0]/counts[1]:.1f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Distribusi Kelas Target — Class Imbalance', fontsize=13, fontweight='bold')

# Bar chart
bars = axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_ylabel('Jumlah Nasabah')
axes[0].set_title('Jumlah per Kelas')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
                 f'{count:,}\n({count/len(df_raw)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylim(0, counts.max() * 1.25)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight('bold')
axes[1].set_title('Proporsi Kelas')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'class_distribution.png'), bbox_inches='tight')
plt.show()
print('✅ Tersimpan: class_distribution.png')

---
## 📊 Sel 3C — EDA: Distribusi Fitur Numerik Kunci

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Distribusi Fitur Numerik Utama', fontsize=13, fontweight='bold')

fitur_plot = [
    ('LIMIT_BAL', 'Batas Kredit (NT$)'),
    ('AGE',       'Usia (tahun)'),
    ('BILL_AMT1', 'Tagihan Bulan Terakhir (NT$)'),
    ('PAY_AMT1',  'Pembayaran Bulan Terakhir (NT$)'),
    ('PAY_0',     'Status Bayar Bulan Terakhir'),
    ('PAY_AMT2',  'Pembayaran 2 Bulan Lalu (NT$)'),
]

for ax, (col, title) in zip(axes.flatten(), fitur_plot):
    for val, color, label in [(0, '#1d7a4a', 'Tidak Gagal Bayar'), (1, '#b3261e', 'Gagal Bayar')]:
        subset = df_raw[df_raw['TARGET'] == val][col]
        ax.hist(subset, bins=30, color=color, alpha=0.55, label=label, density=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Densitas')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 📊 Sel 3D — EDA: Fitur Kategorikal vs Target

In [ ]:
cat_map = {
    'SEX':       {1: 'Laki-laki', 2: 'Perempuan'},
    'EDUCATION': {1: 'Pasca', 2: 'Universitas', 3: 'SMA', 4: 'Lainnya'},
    'MARRIAGE':  {1: 'Menikah', 2: 'Lajang', 3: 'Lainnya'},
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Tingkat Gagal Bayar per Kategori', fontsize=13, fontweight='bold')

for ax, (col, mapping) in zip(axes, cat_map.items()):
    tmp = df_raw.copy()
    tmp[col] = tmp[col].map(mapping).fillna('Lainnya')
    rate = tmp.groupby(col)['TARGET'].mean().sort_values(ascending=False)
    bars = ax.bar(rate.index, rate.values * 100,
                  color=['#b3261e' if v > 0.22 else '#1d7a4a' for v in rate.values],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(col)
    ax.set_ylabel('Tingkat Gagal Bayar (%)')
    ax.set_ylim(0, 40)
    for bar, v in zip(bars, rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v*100:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 📊 Sel 3E — EDA: Heatmap Korelasi

In [ ]:
numeric_cols_eda = ['LIMIT_BAL', 'AGE',
                    'PAY_0', 'PAY_2', 'PAY_3',
                    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
                    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'TARGET']

corr = df_raw[numeric_cols_eda].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
    center=0, linewidths=.5, ax=ax, square=True, cbar_kws={'shrink': .8}
)
ax.set_title('Heatmap Korelasi Fitur Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Korelasi tertinggi dengan TARGET
print('\nKorelasi fitur dengan TARGET (diurutkan):')
print(corr['TARGET'].drop('TARGET').abs().sort_values(ascending=False).to_string())

---
## 🔧 Sel 4 — Pra-pemrosesan Data (Preprocessing)

In [ ]:
# ── Definisi kolom ────────────────────────────────────────────────────────────
NUMERIC_COLS = [
    'LIMIT_BAL', 'AGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1',  'PAY_AMT2',  'PAY_AMT3',  'PAY_AMT4',  'PAY_AMT5',  'PAY_AMT6',
]

CATEGORICAL_LEVELS = {
    'SEX':       [1, 2],
    'EDUCATION': [1, 2, 3, 4],
    'MARRIAGE':  [1, 2, 3],
}

# ── Bersihkan nilai kategorikal out-of-range ───────────────────────────────────
df = df_raw.copy()
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE']  = df['MARRIAGE'].replace({0: 3})

print('Nilai unik EDUCATION setelah bersih:', sorted(df['EDUCATION'].unique()))
print('Nilai unik MARRIAGE  setelah bersih:', sorted(df['MARRIAGE'].unique()))
print('Missing values total :', df.isnull().sum().sum())

# ── Build fitur: numerik + one-hot encoding (drop-first) ─────────────────────
def build_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data[NUMERIC_COLS].astype(float).copy()
    for col, levels in CATEGORICAL_LEVELS.items():
        for v in levels[1:]:                              # drop-first
            out[f'{col}_{v}'] = (data[col].astype(int) == v).astype(float)
    return out

X = build_features(df)
y = df['TARGET'].astype(int)

FEATURE_COLUMNS = list(X.columns)
DUMMY_COLS      = [c for c in FEATURE_COLUMNS if c not in NUMERIC_COLS]

print(f'\nJumlah total fitur : {len(FEATURE_COLUMNS)}')
print(f'  Numerik          : {len(NUMERIC_COLS)}')
print(f'  Dummy kategorikal: {len(DUMMY_COLS)} → {DUMMY_COLS}')
X.head()

In [ ]:
# ── Train-Test Split (80/20, stratified) ──────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Data latih : {len(X_train):,} baris')
print(f'Data uji   : {len(X_test):,} baris')
print()
print('Distribusi TARGET pada data latih:')
print(y_train.value_counts(normalize=True).rename({0:'Tidak Gagal',1:'Gagal Bayar'}).round(3).to_string())
print('\nDistribusi TARGET pada data uji:')
print(y_test.value_counts(normalize=True).rename({0:'Tidak Gagal',1:'Gagal Bayar'}).round(3).to_string())

In [ ]:
# ── StandardScaler (hanya pada kolom numerik) ─────────────────────────────────
scaler = StandardScaler()
scaler.fit(X_train[NUMERIC_COLS])

def scale(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()
    f[NUMERIC_COLS] = scaler.transform(f[NUMERIC_COLS])
    return f

X_train_s = scale(X_train)
X_test_s  = scale(X_test)

print('✅ StandardScaler di-fit pada data latih dan diterapkan ke train & test')
print(f'   Contoh mean scaler (5 pertama): {scaler.mean_[:5].round(2)}')

---
## ⚖️ Sel 5 — Penanganan Class Imbalance dengan SMOTE

In [ ]:
print('Distribusi sebelum SMOTE:')
print(pd.Series(y_train).value_counts().rename({0:'Tidak Gagal Bayar', 1:'Gagal Bayar'}).to_string())

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_s, y_train)

print('\nDistribusi setelah SMOTE (data latih saja):')
print(pd.Series(y_res).value_counts().rename({0:'Tidak Gagal Bayar', 1:'Gagal Bayar'}).to_string())
print(f'\nJumlah sampel: {len(y_train):,} → {len(y_res):,} (+{len(y_res)-len(y_train):,} sampel sintetis)')

In [ ]:
# Visualisasi perubahan distribusi sebelum vs sesudah SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Distribusi Data Latih: Sebelum vs Sesudah SMOTE', fontsize=12, fontweight='bold')

colors = ['#1d7a4a', '#b3261e']
labels = ['Tidak Gagal Bayar', 'Gagal Bayar']

# Sebelum
before = pd.Series(y_train).value_counts().sort_index()
axes[0].bar(labels, before.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Sebelum SMOTE')
axes[0].set_ylabel('Jumlah Sampel')
for i, v in enumerate(before.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_ylim(0, before.max() * 1.2)

# Sesudah
after = pd.Series(y_res).value_counts().sort_index()
axes[1].bar(labels, after.values, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Sesudah SMOTE')
axes[1].set_ylabel('Jumlah Sampel')
for i, v in enumerate(after.values):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
axes[1].set_ylim(0, after.max() * 1.2)

plt.tight_layout()
plt.show()

---
## 🤖 Sel 6 — Training Model Regresi Logistik

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
model = LogisticRegression(
    max_iter  = 1000,
    solver    = 'lbfgs',   # optimizer, cocok untuk data sedang
    random_state = 42,
)

model.fit(X_res, y_res)

print('✅ Model Regresi Logistik selesai dilatih!')
print(f'   Solver         : {model.solver}')
print(f'   Iterasi        : {model.n_iter_[0]} (dari maks {model.max_iter})')
print(f'   Jumlah kelas   : {model.classes_}')
print(f'   Jumlah fitur   : {model.n_features_in_}')

---
## 📏 Sel 7A — Evaluasi: Confusion Matrix

In [ ]:
# ── Prediksi pada data uji ────────────────────────────────────────────────────
THRESHOLD = 0.5
y_proba   = model.predict_proba(X_test_s)[:, 1]
y_pred    = (y_proba >= THRESHOLD).astype(int)

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=' * 50)
print('CONFUSION MATRIX')
print('=' * 50)
print(f'  TN (True Negative)  = {tn:,}  → Nasabah baik, diprediksi baik  ✅')
print(f'  FP (False Positive) = {fp:,} → Nasabah baik, diprediksi berisiko ⚠️')
print(f'  FN (False Negative) = {fn:,}  → Nasabah berisiko, diprediksi baik ❌  ← BERBAHAYA!')
print(f'  TP (True Positive)  = {tp:,}  → Nasabah berisiko, diprediksi berisiko ✅')
print()
print('💡 Interpretasi Bisnis:')
print('   False Negative lebih merugikan perusahaan karena bank tidak')
print('   mengantisipasi nasabah yang benar-benar akan gagal bayar.')
print('   Oleh karena itu RECALL untuk kelas Gagal Bayar menjadi metrik prioritas.')

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Tidak Berisiko', 'Gagal Bayar'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix – Data Uji', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'confusion_matrix.png'), bbox_inches='tight')
plt.show()
print('✅ Tersimpan: confusion_matrix.png')

---
## 📏 Sel 7B — Evaluasi: Precision, Recall, F1-Score

In [ ]:
print('=' * 55)
print('CLASSIFICATION REPORT')
print('=' * 55)
report = classification_report(
    y_test, y_pred,
    target_names=['Tidak Berisiko (0)', 'Gagal Bayar (1)'],
    output_dict=False
)
print(report)

report_dict = classification_report(
    y_test, y_pred,
    target_names=['Tidak Berisiko', 'Gagal Bayar'],
    output_dict=True
)

print('💡 Penjelasan Metrik untuk Manajemen Risiko:')
print(f'   Precision Gagal Bayar : {report_dict["Gagal Bayar"]["precision"]:.4f}')
print(f'      → Dari yang diprediksi berisiko, berapa % benar-benar berisiko')
print(f'   Recall Gagal Bayar  ⭐: {report_dict["Gagal Bayar"]["recall"]:.4f}')
print(f'      → Dari nasabah yang benar berisiko, berapa % berhasil dideteksi')
print(f'   F1-Score Gagal Bayar  : {report_dict["Gagal Bayar"]["f1-score"]:.4f}')
print(f'      → Harmonic mean Precision & Recall')
print(f'   Accuracy              : {report_dict["accuracy"]:.4f}')

---
## 📏 Sel 7C — Evaluasi: AUC-ROC Curve

In [ ]:
auc = roc_auc_score(y_test, y_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

print(f'AUC-ROC Score : {auc:.4f}')
print(f'Interpretasi  : Model {"BAIK" if auc > 0.7 else "CUKUP"} dalam membedakan kelas')
print(f'               (AUC 1.0 = sempurna, AUC 0.5 = tebakan acak)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Analisis ROC & Precision-Recall', fontsize=12, fontweight='bold')

# ROC Curve
axes[0].plot(fpr, tpr, color='#1f4e8c', lw=2, label=f'ROC Curve (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#1f4e8c')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.02])

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec, prec, color='#b3261e', lw=2, label='Precision-Recall')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', lw=1,
                label=f'Baseline ({y_test.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'roc_curve.png'), bbox_inches='tight')
plt.show()
print('✅ Tersimpan: roc_curve.png')

---
## 📏 Sel 7D — Evaluasi: Pentingnya Fitur (Koefisien Regresi Logistik)

In [ ]:
coef_df = pd.DataFrame({
    'Fitur'         : FEATURE_COLUMNS,
    'Koefisien'     : model.coef_[0],
    'AbsKoefisien'  : np.abs(model.coef_[0]),
}).sort_values('AbsKoefisien', ascending=False)

print('10 Fitur Terpenting:')
print(coef_df.head(10)[['Fitur', 'Koefisien']].to_string(index=False))

# Plot
top20 = coef_df.sort_values('AbsKoefisien', ascending=True).tail(20)
colors = ['#b3261e' if c > 0 else '#1d7a4a' for c in top20['Koefisien']]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20['Fitur'], top20['Koefisien'], color=colors)
ax.axvline(0, color='black', lw=0.8, linestyle='--')
ax.set_xlabel('Koefisien (Log-Odds)', fontsize=11)
ax.set_title('20 Fitur Terpenting – Regresi Logistik\n'
             '(merah = meningkatkan risiko  |  hijau = mengurangi risiko)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'feature_importance.png'), bbox_inches='tight')
plt.show()
print('✅ Tersimpan: feature_importance.png')

---
## 📏 Sel 7E — Evaluasi: Analisis Threshold (Optimal)

In [ ]:
# Cari threshold optimal berdasarkan F1-Score kelas Gagal Bayar
thresholds_scan = np.arange(0.1, 0.9, 0.05)
records = []

for t in thresholds_scan:
    pred_t = (y_proba >= t).astype(int)
    rpt    = classification_report(y_test, pred_t, output_dict=True, zero_division=0)
    records.append({
        'threshold' : round(t, 2),
        'precision' : rpt.get('Gagal Bayar', rpt.get('1', {})).get('precision', 0),
        'recall'    : rpt.get('Gagal Bayar', rpt.get('1', {})).get('recall', 0),
        'f1'        : rpt.get('Gagal Bayar', rpt.get('1', {})).get('f1-score', 0),
        'accuracy'  : rpt['accuracy'],
    })

scan_df = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(scan_df['threshold'], scan_df['precision'], label='Precision', marker='o', markersize=4)
ax.plot(scan_df['threshold'], scan_df['recall'],    label='Recall',    marker='s', markersize=4)
ax.plot(scan_df['threshold'], scan_df['f1'],        label='F1-Score',  marker='^', markersize=4, lw=2)
ax.axvline(THRESHOLD, color='gray', linestyle='--', lw=1.5, label=f'Threshold saat ini ({THRESHOLD})')
ax.set_xlabel('Threshold')
ax.set_ylabel('Skor')
ax.set_title('Precision / Recall / F1 vs Threshold (Kelas Gagal Bayar)', fontweight='bold')
ax.legend()
ax.set_xlim([0.1, 0.85])
plt.tight_layout()
plt.show()

best_row = scan_df.loc[scan_df['f1'].idxmax()]
print(f'\nThreshold terbaik untuk F1-Score Gagal Bayar: {best_row["threshold"]}')
print(f'  Precision : {best_row["precision"]:.4f}')
print(f'  Recall    : {best_row["recall"]:.4f}')
print(f'  F1-Score  : {best_row["f1"]:.4f}')

---
## 💾 Sel 8 — Simpan Model, Scaler, dan Metadata

In [ ]:
# ── Simpan model & scaler ─────────────────────────────────────────────────────
joblib.dump(model,  os.path.join(MODEL_DIR, 'model.pkl'))
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))
print('✅ model.pkl  → tersimpan')
print('✅ scaler.pkl → tersimpan')

# ── Simpan metadata ───────────────────────────────────────────────────────────
meta = {
    'features'   : FEATURE_COLUMNS,
    'numeric'    : NUMERIC_COLS,
    'threshold'  : THRESHOLD,
    'smote_used' : True,
    'auc'        : round(float(auc), 4),
    'n_train'    : int(len(X_train)),
    'n_test'     : int(len(X_test)),
    'metrics': {
        'precision_gagal_bayar' : round(report_dict['Gagal Bayar']['precision'], 4),
        'recall_gagal_bayar'    : round(report_dict['Gagal Bayar']['recall'],    4),
        'f1_gagal_bayar'        : round(report_dict['Gagal Bayar']['f1-score'],  4),
        'precision_tidak'       : round(report_dict['Tidak Berisiko']['precision'], 4),
        'recall_tidak'          : round(report_dict['Tidak Berisiko']['recall'],    4),
        'f1_tidak'              : round(report_dict['Tidak Berisiko']['f1-score'],  4),
        'accuracy'              : round(report_dict['accuracy'], 4),
    },
    'confusion_matrix': {
        'TN': int(tn), 'FP': int(fp),
        'FN': int(fn), 'TP': int(tp),
    },
}

with open(os.path.join(MODEL_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('✅ meta.json  → tersimpan')

print(f'\n📁 Semua file tersimpan di folder: {MODEL_DIR}/')
for fname in os.listdir(MODEL_DIR):
    fpath = os.path.join(MODEL_DIR, fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f'   {fname:<20} {size:.1f} KB')

print(f'\n📁 Grafik evaluasi tersimpan di: {EVAL_DIR}/')
for fname in os.listdir(EVAL_DIR):
    print(f'   {fname}')

---
## 🧪 Sel 9 — Uji Prediksi Manual (Contoh Nasabah)

In [ ]:
def predict_nasabah(data: dict, threshold: float = THRESHOLD) -> dict:
    """Buat prediksi untuk satu nasabah baru."""
    df_input = pd.DataFrame([data])
    X_input  = build_features(df_input)
    X_input[NUMERIC_COLS] = scaler.transform(X_input[NUMERIC_COLS])
    proba = float(model.predict_proba(X_input)[0, 1])
    return {
        'probabilitas_gagal_bayar': round(proba, 4),
        'persentase'              : f'{proba*100:.1f}%',
        'prediksi'                : 'Berisiko Gagal Bayar' if proba >= threshold else 'Tidak Berisiko',
        'level_risiko'            : 'Tinggi' if proba >= 0.5 else ('Sedang' if proba >= 0.3 else 'Rendah'),
    }

# ── Contoh 1: Nasabah berisiko tinggi ─────────────────────────────────────────
nasabah_1 = {
    'LIMIT_BAL': 20000, 'AGE': 24,
    'SEX': 2, 'EDUCATION': 2, 'MARRIAGE': 1,
    'PAY_0': 2, 'PAY_2': 2, 'PAY_3': -1, 'PAY_4': -1, 'PAY_5': -2, 'PAY_6': -2,
    'BILL_AMT1': 3913, 'BILL_AMT2': 3102, 'BILL_AMT3': 689,
    'BILL_AMT4': 0,    'BILL_AMT5': 0,    'BILL_AMT6': 0,
    'PAY_AMT1': 0,    'PAY_AMT2': 689,   'PAY_AMT3': 0,
    'PAY_AMT4': 0,    'PAY_AMT5': 0,     'PAY_AMT6': 0,
}

print('Contoh 1 – Nasabah Berisiko Tinggi:')
hasil_1 = predict_nasabah(nasabah_1)
for k, v in hasil_1.items():
    print(f'  {k:<35}: {v}')

print()

# ── Contoh 2: Nasabah aman ────────────────────────────────────────────────────
nasabah_2 = {
    'LIMIT_BAL': 500000, 'AGE': 45,
    'SEX': 1, 'EDUCATION': 1, 'MARRIAGE': 1,
    'PAY_0': -1, 'PAY_2': -1, 'PAY_3': -1, 'PAY_4': -1, 'PAY_5': -1, 'PAY_6': -1,
    'BILL_AMT1': 25000, 'BILL_AMT2': 22000, 'BILL_AMT3': 20000,
    'BILL_AMT4': 18000, 'BILL_AMT5': 15000, 'BILL_AMT6': 12000,
    'PAY_AMT1': 25000, 'PAY_AMT2': 22000, 'PAY_AMT3': 20000,
    'PAY_AMT4': 18000, 'PAY_AMT5': 15000, 'PAY_AMT6': 12000,
}

print('Contoh 2 – Nasabah Tidak Berisiko:')
hasil_2 = predict_nasabah(nasabah_2)
for k, v in hasil_2.items():
    print(f'  {k:<35}: {v}')

---
## 📋 Sel 10 — Ringkasan Akhir

In [ ]:
print('=' * 60)
print('  RINGKASAN HASIL TA-03')
print('=' * 60)
print()
print('📊 Dataset')
print(f'   Total data  : {len(df_raw):,} nasabah')
print(f'   Jumlah fitur: {len(FEATURE_COLUMNS)} (20 numerik + 6 dummy)')
print()
print('⚙️  Model')
print( '   Algoritma   : Logistic Regression')
print( '   Sampling    : SMOTE (pada data latih)')
print( '   Scaling     : StandardScaler')
print(f'   Threshold   : {THRESHOLD}')
print()
print('📈 Performa (data uji)')
print(f'   AUC-ROC                   : {auc:.4f}')
print(f'   Accuracy                  : {report_dict["accuracy"]:.4f}')
print(f'   Recall    Gagal Bayar ⭐  : {report_dict["Gagal Bayar"]["recall"]:.4f}')
print(f'   Precision Gagal Bayar     : {report_dict["Gagal Bayar"]["precision"]:.4f}')
print(f'   F1-Score  Gagal Bayar     : {report_dict["Gagal Bayar"]["f1-score"]:.4f}')
print()
print('💡 Kesimpulan Bisnis')
print('   Recall Gagal Bayar ~63% artinya model mampu mendeteksi')
print('   sekitar 63% nasabah yang akan benar-benar gagal bayar.')
print('   Ini mengurangi paparan risiko bank secara signifikan.')
print('   False Negative (489 kasus) perlu mendapat perhatian lebih.')
print('   Menurunkan threshold ke ~0.4 dapat meningkatkan Recall')
print('   namun akan menaikkan False Positive.')
print()
print('💾 Output')
print(f'   model/model.pkl    — model terlatih')
print(f'   model/scaler.pkl   — StandardScaler')
print(f'   model/meta.json    — metadata & metrik')
print(f'   model/eval/*.png   — 4 grafik evaluasi')
print()
print('🌐 Jalankan Aplikasi Web:')
print('   python app.py  →  http://localhost:5000')
print('=' * 60)